# PoliMillionaire baseline

Before running this notebook, put the files in Google Drive like this:

```
MyDrive/
└── Colab Notebooks/
    └── NLP_assignment/
        ├── poli_millionaire_clean_baseline_v2.ipynb
        └── millionaire_client/
            ├── __init__.py
            ├── client.py
            ├── auth.py
            ├── base.py
            ├── game.py
            ├── models.py
            ├── competitions.py
            ├── leaderboard.py
            └── exceptions.py
```

In [1]:
from google.colab import drive
drive.mount('/content/gdrive/')

import os
import sys
import time
import re
import torch

Mounted at /content/gdrive/


In [2]:
BASE_DIR = '/content/gdrive/MyDrive/NLP_assignment'
PACKAGE_DIR = os.path.join(BASE_DIR, 'millionaire_client')

print('BASE_DIR exists:', os.path.exists(BASE_DIR))
if os.path.exists(BASE_DIR):
    print('BASE_DIR contents:', os.listdir(BASE_DIR))

print('PACKAGE_DIR exists:', os.path.exists(PACKAGE_DIR))
if os.path.exists(PACKAGE_DIR):
    print('PACKAGE_DIR contents:', os.listdir(PACKAGE_DIR))

if not os.path.exists(BASE_DIR):
    raise FileNotFoundError('BASE_DIR not found. Create the NLP_assignment folder in Drive and upload the notebook there.')

if not os.path.exists(PACKAGE_DIR):
    raise FileNotFoundError('millionaire_client folder not found inside BASE_DIR.')

if BASE_DIR not in sys.path:
    sys.path.append(BASE_DIR)

print('Path added successfully.')

BASE_DIR exists: True
BASE_DIR contents: ['PoliMillionaire.ipynb', '.DS_Store', 'millionaire_client', '.ipynb_checkpoints', 'test3_rag_game_runs', 'test3_offline_rag_index.pkl', 'pdfs']
PACKAGE_DIR exists: True
PACKAGE_DIR contents: ['base.py', 'leaderboard.py', 'auth.py', 'competitions.py', 'client.py', 'game.py', '__init__.py', 'exceptions.py', 'models.py', '__pycache__']
Path added successfully.


In [3]:
!pip install -q transformers accelerate bitsandbytes sentencepiece protobuf faiss-cpu sentence-transformers pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 95.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 83.5 MB/s eta 0:00:00:00:0100:01


In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from millionaire_client import MillionaireClient
from millionaire_client.exceptions import TimeoutError, RateLimitError

In [5]:
API_URL = 'http://131.175.15.22:51111/'
USERNAME = 'gary'
PASSWORD = '13790229'

client = MillionaireClient(API_URL)
user = client.login(USERNAME, PASSWORD)
print('Logged in as:', user.username)

Logged in as: gary


In [6]:
competitions = client.competitions.list_all()
for c in competitions:
    print(c.id, c.name, c.max_levels)

COMPETITION_ID = competitions[3].id

0 Entertainment 15
1 Ancient History and Politics 15
2 Science and Nature 15
3 Maths 15
4 Philosophy and Psychology 15
5 News 15


In [7]:
model_id = 'Qwen/Qwen2.5-Math-1.5B-Instruct'

# Reverted strictly back to the optimized 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Added attn_implementation="sdpa" to speed up attention calculation overhead
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map='auto',
    attn_implementation="sdpa" 
)
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/656 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.32k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear4bit(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear4bit(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1

In [8]:
import fitz # PyMuPDF
import faiss
from sentence_transformers import SentenceTransformer
import glob
import re

# Try to load embeddings model (lightweight, fits in colab alongside 7B easily)
embedder = SentenceTransformer('all-MiniLM-L6-v2')

def clean_extracted_text(raw_text):
    # Remove null bytes and non-printable characters
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\xff]', '', raw_text)
    # Replace multiple whitespace characters (including newlines) with a single space
    text = re.sub(r'\s+', ' ', text)
    # Strip leading/trailing spaces
    return text.strip()

# Look for PDFs in the base directory or a specific pdfs folder
pdf_dir = os.path.join(BASE_DIR, 'pdfs')
if not os.path.exists(pdf_dir):
    os.makedirs(pdf_dir)
    print(f"Created PDF directory at {pdf_dir}. Please upload PDFs here.")

pdf_files = glob.glob(os.path.join(pdf_dir, '*.pdf'))
chunks = []
chunk_size = 500

if pdf_files:
    print(f"Found {len(pdf_files)} PDFs. Building RAG index...")
    for pdf_file in pdf_files:
        doc = fitz.open(pdf_file)
        text = ""
        for page in doc:
            text += page.get_text() + " "
            
        # Clean the text to remove strange formatting, hidden characters, and huge gaps
        text = clean_extracted_text(text)
        
        # Simple word-based chunking
        words = text.split()
        for i in range(0, len(words), chunk_size):
            chunks.append(" ".join(words[i:i+chunk_size]))

    if chunks:
        # Create embeddings and build FAISS index
        embeddings = embedder.encode(chunks, convert_to_numpy=True)
        dimension = embeddings.shape[1]
        index = faiss.IndexFlatL2(dimension)
        index.add(embeddings)
        print(f"Successfully built FAISS index with {len(chunks)} chunks.")
    else:
        print("No text found in PDFs.")
        index = None
else:
    print("No PDFs found. Skipping RAG indexing.")
    index = None

def retrieve_context(query, k=3):
    if not chunks or index is None:
        return ""
    
    query_vector = embedder.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_vector, k)
    
    retrieved = []
    for idx in indices[0]:
        if idx < len(chunks):
            retrieved.append(chunks[idx])
            
    return "\n...\n".join(retrieved)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Found 18 PDFs. Building RAG index...
Successfully built FAISS index with 11240 chunks.


In [28]:
import time
from transformers import StoppingCriteria, StoppingCriteriaList

MAX_TOKENS = 300

class TimeLimitCriteria(StoppingCriteria):
    def __init__(self, start_time, max_seconds):
        self.start_time = start_time
        self.max_seconds = max_seconds
        
    def __call__(self, input_ids, scores, **kwargs):
        return time.time() - self.start_time > self.max_seconds

def extract_letter(text):
    # If the wrap-up block triggered, check specifically at the end of the text
    wrap_up_idx = text.rfind("[TIME/TOKEN LIMIT REACHED: Forced Wrap-up] ->")
    if wrap_up_idx != -1:
        # Search for A, B, C, D in the wrapped-up portion only
        wrap_match = re.search(r'\b([ABCD])\b', text[wrap_up_idx:].upper())
        if wrap_match:
            return wrap_match.group(1)

    # Try to find a specific final answer indicator, spanning across newlines if necessary
    match = re.search(r'(?:final answer|answer is|correct option .*?is).*?\b([ABCD])\b', text, re.IGNORECASE | re.DOTALL)
    if match:
        return match.group(1).upper()
    
    # Fallback: get all standalone A, B, C, or D occurrences and take the LAST one
    matches = re.findall(r'\b([ABCD])\b', text.upper())
    if matches:
        return matches[-1]
        
    return 'A'

def choose_answer(question):
    if len(question.options) < 4:
        return question.options[0].id, 'A', 'fallback'

    context = retrieve_context(question.text, k=3) # Increased k to 5 for potentially richer context
    context_block = f"\n[Supportive Context (Use this to prevent hallucination in reasoning, but rely on your own knowledge and generate easily if you already know the problem)]:\n{context}\n\n" if context else ""

    prompt_text = f'''PoliMillionaire MCQ.{context_block}Reason through the problem step by step (do not write too long due to token limits which is {MAX_TOKENS} tokens), and then provide your final answer as a single letter (A, B, C, or D).

Question: {question.text}
A) {question.options[0].text}
B) {question.options[1].text}
C) {question.options[2].text}
D) {question.options[3].text}'''

    messages = [
        {"role": "system", "content": "You are a helpful AI assistant playing a trivia game. You must eventually provide a single letter A, B, C, or D as your final answer."},
        {"role": "user", "content": prompt_text}
    ]
    
    # Qwen2.5 Instruct requires chat templates to format properly
    prompt = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    input_len = inputs['input_ids'].shape[1]

    # Stop early if it takes more than 25 seconds
    start_time = time.time()
    time_criteria = TimeLimitCriteria(start_time, 25.0)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_TOKENS,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            stopping_criteria=StoppingCriteriaList([time_criteria])
        )

    text = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
    
    # Check if we were interrupted by the 25-second limit or the 400 token limit
    elapsed_time = time.time() - start_time
    generated_tokens = outputs.shape[1] - input_len
    
    if elapsed_time >= 22.0 or generated_tokens >= MAX_TOKENS - 1:
        # Inject the wrap-up message and force the LLM to output just the final letter!
        full_text_so_far = tokenizer.decode(outputs[0], skip_special_tokens=False) # Keep chat tokens for continuity
        
        # Qwen-specific forced instruction injection
        wrap_up_prompt = full_text_so_far + "\n<|im_start|>user\nWait, I am running out of time. So, without further reasoning, the correct option letter (A, B, C, or D) is:<|im_end|>\n<|im_start|>assistant\nThe correct option is "
        
        wrap_inputs = tokenizer(wrap_up_prompt, return_tensors='pt').to(model.device)
        wrap_input_len = wrap_inputs['input_ids'].shape[1]
        
        with torch.no_grad():
            wrap_outputs = model.generate(
                **wrap_inputs,
                max_new_tokens=5, # Only need 5 tokens to output " A."
                temperature=0.1,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
            
        wrap_text = tokenizer.decode(wrap_outputs[0][wrap_input_len:], skip_special_tokens=True)
        text = text + "\n\n[TIME/TOKEN LIMIT REACHED: Forced Wrap-up] -> " + wrap_text

    letter = extract_letter(text)
    idx = ['A', 'B', 'C', 'D'].index(letter)
    return question.options[idx].id, letter, text, context

In [29]:
def play_game():
    game = client.game.start(competition_id=COMPETITION_ID, mode='text')

    while game.in_progress:
        question = game.current_question
        if question is None:
            break

        print('Level:', game.current_level)
        print(question.text)
        for i, opt in enumerate(question.options):
            print(f"{chr(65+i)}) {opt.text}")

        # extract_letter reverted, so we pass just (question) to choose_answer
        option_id, letter, raw, context = choose_answer(question)
        if context:
            print('\n--- Retrieved Chunks ---\n' + context + '\n------------------------\n')
        print('Predicted:', letter, '| Raw output:', raw)

        max_retries = 5
        for attempt in range(max_retries):
            try:
                result = game.answer(option_id)
                break
            except RateLimitError:
                wait_time = 10 * (attempt + 1)  # Increase the wait time gradually
                print(f'Rate limited, waiting {wait_time} seconds...')
                time.sleep(wait_time)
            except TimeoutError:
                print('Timed out')
                break
        else:
            print('Max retries exceeded for rate limit.')
            break
            
        if 'result' not in locals() or result is None:
            break

        print('Correct:', result.correct, '| Earned:', result.earned_amount)

        if result.game_over:
            break

        time.sleep(0.5)

    return print('Final earned:', game.earned_amount)

In [30]:
play_game()

Level: 1
Find $-\dfrac{1}{-3}\cdot\cfrac{1}{~\frac{1}{-3}~}.$
A) 2
B) 0
C) -1
D) 1

--- Retrieved Chunks ---
3 - 4 + 5 - 6 + g + (-1)n+1n + g (3) 10.6Alternating Series and Conditional Convergence 611 We see from these examples that the nth term of an alternating series is of the form an = (-1)n+1un or an = (-1)nun where un = an is a positive number. Series (1), called the alternating harmonic series, converges, as we will see in a moment. Series (2), a geometric series with ratio r = -1>2, converges to -2> 31 + (1>2)4 = -4>3. Series (3) diverges because the nth term does not approach zero. We prove the convergence of the alternating harmonic series by applying the Alternat- ing Series Test. This test is for convergence of an alternating series and cannot be used to conclude that such a series diverges. The test is also valid for the alternating series -u1 + u2 - u3 + g, like the one in Series (2) given above. THEOREM 15—The Alternating Series Test The series a q n=1 (-1)n+1un = u1 - u

In [ ]:
''' --- IGNORE ---
import gc
import torch

try:
    del model
    del tokenizer
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()
print("CUDA VRAM emptied.")
''' 

' --- IGNORE ---\nimport gc\nimport torch\n\ntry:\n    del model\n    del tokenizer\nexcept NameError:\n    pass\n\ngc.collect()\ntorch.cuda.empty_cache()\nprint("CUDA VRAM emptied.")\n'

In [ ]:
import torch
free, total = torch.cuda.mem_get_info()
print("Free GB:", free / 1024**3)
print("Total GB:", total / 1024**3)
print("Allocated GB:", torch.cuda.memory_allocated() / 1024**3)
print("Reserved GB:", torch.cuda.memory_reserved() / 1024**3)

Free GB: 2.24786376953125
Total GB: 14.56317138671875
Allocated GB: 4.489849090576172
Reserved GB: 12.1875
